# Phase 2 — Feature Engineering

**Goal:** Parse raw string columns into numeric/boolean features, standardise processor names, and export the model-ready CSV.

> **Data leakage note:** The `rating` column (post-sale review score) is dropped before saving so it cannot leak into model training.

**Input:** `data/Clean1_Smart_phone.csv`  
**Output:** `data/Clean2_Smart_Phones.csv`

## 1. Import Libraries

### Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import warnings


## 2. Load Data & Rearrange Brand Column

In [ ]:
warnings.filterwarnings("ignore")
df = pd.read_csv("data\Clean1_Smart_phone.csv")
pd.set_option('display.max_rows', None)
df.head(2)


In [ ]:
df.rename(columns={"Company":"Brand"},inplace=True)
Brand  = df["Brand"]
df.drop(columns=["Unnamed: 0","Brand"],inplace=True)


### Insert at specific pos

In [ ]:
df.insert(0,"Brand",Brand)
df["Brand"].value_counts().head(5)

## 3. SIM Column → Boolean Feature Flags

Extract `Has_5G`, `Has_NFC`, `Has_IR_Blaster` from the free-text `sim` column, then drop `sim` (the raw string is no longer needed).

In [ ]:
df["sim"].value_counts().head(10)


In [ ]:
Has_5G  = df["sim"].str.contains("5G")
Has_NFC  = df["sim"].str.contains("NFC")
Has_IR_Blaster  = df["sim"].str.contains("IR Blaster")

df.insert(4,"Has_5G",Has_5G)
df.insert(5,"Has_NFC",Has_NFC)
df.insert(6,"Has_IR_Blaster",Has_IR_Blaster)
df.drop(columns="sim",inplace=True)

df.head(3)


## 4. Processor Column Cleanup

The raw `processor` string contains the full description. We:
1. Fill in known incomplete entries manually
2. Strip vendor prefixes (`Qualcomm`, `Google`, `Samsung`, `Apple`, `Tiger`)
3. Standardise manufacturer brand names
4. Parse into 4 derived columns: `Processor_Model`, `Processor_Name`, `Processor_Core`, `Processor_Speed`

In [ ]:
df["processor"].value_counts().head(10)


### 4.1 Fill Incomplete Processor Names

In [ ]:
print(df[df["processor"].str.split(",").str.get(0).str.contains("Core")][["index","model","processor"]])
processor_mapping =  {
    98: "Tesla Custom Chip, Octa Core, 3.0 GHz Processor",
    143: "Exynos 2200, Octa Core, 2.8 GHz Processor",
    397: "Snapdragon 778G, Octa Core, 2.4 GHz Processor",
    399: "Snapdragon 730G, Octa Core, 2.2 GHz Processor",
    427: "Exynos 1380, Octa Core, 2.4 GHz Processor",
    455: "Kirin 710, Octa Core, 2.2 GHz Processor",
    596: "Exynos 850, Octa Core, 2.0 GHz Processor",
    627: "Helio G99, Octa Core, 2.2 GHz Processor",
    645: "Exynos 850, Octa Core, 2.0 GHz Processor",
    671: "Exynos 1330, Octa Core, 2.4 GHz Processor",
    691: "Exynos 850, Octa Core, 2.0 GHz Processor",
    748: "Helio G85, Octa Core, 2.0 GHz Processor",
    860: "Snapdragon 450, Octa Core, 1.8 GHz Processor",
    910: "Unisoc T107, Octa Core, 1.6 GHz Processor",
    911: "Unisoc Generic, Quad Core, 1.6 GHz Processor",
     948: "Unisoc SC7731E, Quad Core, 1.3 GHz Processor",
    950: "Unisoc Generic, Octa Core, 1.6 GHz Processor",
    962: "Helio A22, Quad Core, 2.0 GHz Processor",
    973: "Unisoc SC7731E, Quad Core, 1.3 GHz Processor",
    983: "MediaTek Generic, Quad Core, 1.4 GHz Processor"
}



for idx, processor in processor_mapping.items():
    mask = df['index'] == idx
    df.loc[mask, 'processor'] = processor


*Verify — all processors now have a named chip before the first comma:*

In [ ]:
df[df["processor"].str.split(",").str.get(0).str.contains("Core")][["index","model","processor"]]

### 4.2 Clean Vendor-Prefixed Processor Names

In [ ]:

print(df[df["processor"].str.contains("Qualcomm")][["index","model"]])
new_rows_map = {
    614: "Snapdragon 670, Octa Core, 2.0 GHz Processor",
    658: "Snapdragon 215, Quad Core, 1.3 GHz Processor",
    840: "Snapdragon 625, Octa Core, 2.0 GHz Processor",
    929: "Snapdragon QM215, Quad Core, 1.3 GHz Processor",
    981: "Snapdragon QM215, Quad Core, 1.3 GHz Processor"
}

for idx, proc_val in new_rows_map.items():
         df.loc[df["index"] == idx, 'processor'] = proc_val
df[df["processor"].str.contains("Qualcomm")][["index","model"]]


### 4.3 Standardise Chip Manufacturer Names

Map `Spreadtrum` → `Unisoc`, `Apple` → `Bionic`, `Tiger` → `Unisoc`, etc.

In [ ]:

# 1. Clean up "Google " and "Tiger" first as you did
df['processor'] = df['processor'].str.replace('Google ', '', case=False)
df['processor'] = df['processor'].str.replace('Tiger', 'Unisoc', case=False)

# 2. Use a loop to handle the dictionary for substring replacement
final_cleanup = {
    'Spreadtrum': 'Unisoc',
    'Apple': 'Bionic',
    'Fusion': 'Bionic',
    'Samsung': 'Exynos',
    'Mediatek': 'Dimensity' # Standardizing capitalization
}

for old, new in final_cleanup.items():
    # regex=False is faster and safer here since we are just swapping names
    df['processor'] = df['processor'].str.replace(old, new, case=False, regex=False)


*Verify — all processors now have a named chip before the first comma:*

In [ ]:
df.loc[973, 'processor'] = "Snapdragon QM215, Quad Core, 1.3 GHz Processor"

Processer_Model  = df["processor"].str.strip().str.split().str.get(0).str.capitalize()
Processer_Name  = df["processor"].str.split(",").str.get(0).str.split("P").str.get(0).str.strip()
Core = df["processor"].str.split(",").str.get(1).str.split("P").str.get(0).str.strip()
Processer_Speed = df["processor"].str.split(",").str.get(2).str.split("G").str.get(0).str.strip()

df.insert(7,"Processor_Model",Processer_Model)
df.insert(8,"Processor_Name",Processer_Name)

df.insert(9,"Processor_Core",Core)
df.insert(10,"Processor_Speed",Processer_Speed)


In [ ]:
df["Processor_Name"].value_counts().head(10)


### 4.5 Fix Processor Column Misalignment

Rows where the chip model itself contains 'Core' (e.g. Octa Core was parsed as model) need a one-position right-shift across Name/Core/Speed.

In [ ]:
temp_df  = df[df["Processor_Name"].str.contains("Core")][["Processor_Name","Processor_Core","Processor_Speed"]].shift(1,axis=1)


df.loc[temp_df.index,["Processor_Name","Processor_Core","Processor_Speed"]] = temp_df.values

### 4.6 Convert Core Type to Numeric

Map `Octa Core` → 8, `Hexa Core` → 6, `Quad Core` → 4.

In [ ]:
print(df["Processor_Core"].value_counts())



### 4.7 Clean Processor Speed to Float

In [ ]:
print("Some GHz still in string" ,df["Processor_Speed"].value_counts().tail(5))
df["Processor_Speed"] = df["Processor_Speed"].str.split().str.get(0)

df["Processor_Speed"] = df["Processor_Speed"].astype("float64")

print(df["Processor_Speed"].value_counts())
df["Processor_Speed"].describe()

In [ ]:
df["ram"].value_counts().head(5)

# Extract and Convert RAM and Memory (Storage) to Numeric Values

1. **Extract RAM and Memory**: Split the ram column to extract RAM size (first number) and internal storage size (second number after comma), then insert them as new columns at positions 12 and 13.
2. **Convert to numeric**: Transform both Ram and Memory columns from string to float64 data type for numerical operations and analysis.

---

In [ ]:
ram = df["ram"].str.strip().str.split().str.get(0)
memory  = df["ram"].str.strip().str.split(",").str.get(1).str.split().str.get(0).str.strip()


df.insert(12,"Ram",ram)
df.insert(13,"Memory",memory)

df["Ram"] = df["Ram"].astype("float64")
df["Memory"] = df["Memory"].astype("float64")
df.head(2)


In [ ]:
df["battery"].value_counts()

# Fix Misaligned RAM and Memory Data

1. **Identify and shift**: Find rows where Memory is null (indicating RAM data is in the wrong column), then shift both Ram and Memory columns one position to the right to correct the alignment.
2. **Apply correction**: Update the original dataframe with the corrected values for those specific rows and display the fixed data.

In [ ]:

temp_df  = df[df["Memory"].isnull()][["Ram","Memory"]].shift(1,axis=1)
df.loc[temp_df.index,["Ram",'Memory']] = temp_df.values
print(temp_df)

### Manually fill Ram of phone  
1. Huawei == 12GB
2. iphones == 4GB

In [ ]:
df.loc[temp_df.index].head(1)["Ram"].fillna(12,inplace=True)    
df["Ram"].fillna(4,inplace=True)

In [ ]:
print(df.loc[947]) ### Mobile with Ram in MB So we covert it into GB -> MB
df.loc[947,'Ram'] = 512/1024

In [ ]:
df["Ram"].value_counts()

#### Replacing 1 TB Memory with Gb -> 1 TB = 1024 GB

In [ ]:
temp_df  = df[df["Memory"] == 1]
df.loc[temp_df.index,'Memory'] = 1024    
df[df["Memory"] == 1024]

In [ ]:
df["battery"].value_counts().sample(5)

In [ ]:
battery = df["battery"].str.split().str.get(0).str.strip()
Fast_charge = df["battery"].str.split("with").str.get(1).str.strip().str.split("W").str.get(0)
df[Fast_charge == "Fast Charging"]["model"].to_dict()

In [ ]:


# Updated dictionary with correct indices
fast_charging_wattage = {
    # iPhone 14 Series
    5: 27, 7: 25, 8: 27, 17: 27, 23: 25, 24: 27, 29: 25, 
    46: 25, 52: 25, 66: 25, 83: 25, 85: 25, 102: 25, 108: 25,
    
    # iPhone 13 Series
    6: 27, 15: 23, 19: 27, 25: 27, 26: 23, 31: 23, 
    59: 20, 89: 20, 111: 20, 115: 20,
    
    # iPhone 15 Series
    14: 27, 68: 27, 73: 27,
    
    # Other iPhones
    91: 18, 308: 18,
    
    # Samsung Galaxy
    3: 40, 55: 25, 143: 25, 171: 25, 229: 15, 252: 15, 
    287: 25, 397: 25, 399: 25, 407: 25, 427: 25, 
    625: 15,  # CHANGED: was 626 → Samsung Galaxy A15
    734: 15,  # CHANGED: was 733 → Samsung Galaxy A04
    772: 15,  # CHANGED: was 771 → Samsung Galaxy A04
    
    # Google Pixel
    258: 18, 304: 18, 472: 18, 
    629: 18,  # CHANGED: was 628 → Google Pixel 2 XL
    
    # OnePlus
    195: 65, 434: 65,
    
    # Vivo
    376: 25, 398: 44, 
    746: 18,  # CHANGED: was 745 → Vivo Y3
    
    # LG
    77: 25, 144: 25,
    
    # Xiaomi/Poco
    483: 67, 504: 33, 
    736: 33,  # CHANGED: confirmed → Xiaomi Poco F1 (6GB)
    771: 18,
    
    # Motorola
    601: 15,  # CHANGED: was 600 → Motorola Moto G62
    659: 15,  # CHANGED: was 658 → Motorola Moto G62 5G
    
    # OPPO
    646: 18,  # CHANGED: was 645 → OPPO A54
    759: 18,  # CHANGED: was 758 → OPPO A17
    768: 18,  # CHANGED: was 767 → OPPO A54
    792: 18,  # CHANGED: was 790 → OPPO A54
    
    # Others
    27: 30, 128: 27, 
    627: 18,  # CHANGED: was 624 → Sony Xperia L5 5G
    674: 18,  # CHANGED: was 673 → BLU F91 5G
}

In [ ]:
df.insert(15,"Battery",battery)
df.insert(16,"Fast_Charge",Fast_charge)

df["Battery"] = df["Battery"].astype("float64")


In [ ]:
# Replace for specific indices only
for idx, watt in fast_charging_wattage.items():
    mask = df['index'] == idx
    df.loc[mask, 'Fast_Charge'] = df.loc[mask,'Fast_Charge'].str.replace(
        'Fast Charging', 
        f'{watt}'
    )

In [ ]:
df["Fast_Charge"] = df["Fast_Charge"].astype("Float64")
df["Fast_Charge"].fillna(0,inplace=True)

In [ ]:
screen_size  = df["display"].str.strip().str.split(",").str.get(0).str.strip().str.split().str.get(0).astype("float64")
df.insert(16,"Screen_size",screen_size)

In [ ]:
resolution  = df["display"].str.strip().str.split(",").str.get(1).str.strip().str.split("px").str.get(0)
df.insert(17,"Resolution",resolution)

In [ ]:
refresh_rate  = df["display"].str.strip().str.split(",").str.get(2).str.strip().str.split(" ").str.get(0).apply(lambda x : 60 if pd.isna(x) else x).astype("int")
df.insert(18,"Refresh_Rate",refresh_rate)

### Extracting last values like 12MP + 10 MP Dual Rear Camera 
#### Mean Get before value from rear that is Dual
#### If 12 MP Rear Camera mean MP then single camera

In [ ]:
df["camera"]

In [ ]:
rear_Camera  = df["camera"].str.split("Rear").str.get(0).str.strip().str.split(" ").str.get(-1)

df.insert(18,"Num_Back_Cam",rear_Camera)
df["camera"].head(3)

In [ ]:
temp_df = df[df["Num_Back_Cam"].str.contains("MP")]
df.loc[temp_df.index,'Num_Back_Cam'] = "Single"   ### Where there Is MP it have one camera only
df.loc[435,'camera'] = "64 MP + 20 MP + 2 MP Triple Rear & 20 MP Front"  ## ADD Front info


df["Num_Back_Cam"].value_counts()

#### Get how many front cameras 
1. If MP mean 12 MP Front camera ,then 1 front camera
2. If Dual mean 2 camera
3. If Camera it mean no camera it extract last value of back camera

In [ ]:
Num_Front_Cam  = df["camera"].str.split("Front").str.get(0).str.strip().str.split().str.get(-1)
Num_Front_Cam.value_counts()
df.insert(19,"Num_Front_Cam",Num_Front_Cam)


temp_df = df[df["Num_Front_Cam"]=="Camera"]
df.loc[temp_df.index,'Num_Front_Cam'] = "0"
print(temp_df)
print(df["Num_Front_Cam"].value_counts())

df["Num_Front_Cam"]  = df["Num_Front_Cam"].replace({"MP":1,"Dual":2,"0":0},inplace=True)



### Extract Main(Back) Camera first Mega Pixel 
#### if 80 MP + 12 MP + 8 MP Rear camera then 80 value of back camera

In [ ]:
Back_Camera_MP = df["camera"].str.split().str.get(0)
df.insert(19,"Back_Camera_MP",Back_Camera_MP)
df["Back_Camera_MP"] = df["Back_Camera_MP"].astype("float64")
df["Num_Back_Cam"] = df["Num_Back_Cam"].replace({"Single":1,"Dual":2,"Triple":3,"Quad":4,"Penta":5},inplace=True)


### Extract Front Camera Mega Pixel like 24 MP Front camera 
#### Then 24 is MP 

In [ ]:
Front_Cam_MP = df["camera"].str.strip().str.split("&").str.get(1).str.split().str.get(0).astype(float)
df.insert(20,"Front_CAM_MP",Front_Cam_MP)

df["Front_CAM_MP"]  = df["Front_CAM_MP"].fillna(0,inplace=True)

df["Front_CAM_MP"].value_counts().head(5)

#### Some Os value are not available so by gpt get their os information or fill it in missing value

In [ ]:
index_to_os_version = {
    9:   "Android v9",      # Huawei Mate X
    11:  "Android v11",     # Huawei Mate Xs 2
    13:  "Android v12",     # Royole FlexPai 3 5G
    38:  "Android v10",     # Royole FlexPai 2
    41:  "Android v12",     # Asus ROG Phone 6D Ultimate
    53:  "Android v12",     # Oppo Find N2 5G
    61:  "Android v12",     # Samsung Galaxy Z Flip 4 5G
    62:  "Android v12",     # OPPO Find N Flip
    77:  "Android v10",     # LG V60 ThinQ
    95:  "Android v13",     # OPPO Find N2 Flip
    99:  "Android v11",     # Samsung Galaxy Z Flip 3
    144: "Android v10",     # LG Wing 5G
    435: "Android v12",     # Oukitel WP21
    658: "Android v11 Go"   # CAT S22 Flip
}



df['os'] = df['os'].fillna(df['index'].map(index_to_os_version))
    
df["os"] =df["os"].replace({"Bluetooth":"KaiOS v2.5"},inplace= True)

#### These os values are missing so add manually

In [ ]:
df.loc[310,"os"] =  "Android v12"
df.loc[888,'os'] =  "Android v12"
df.loc[927,'os'] = "Android v11 Go"

#### Replace Some os to it proper form ensuring consistency 

In [ ]:
OS_Name  = df["os"].str.split().str.get(0)
df.insert(25,"OS_Name",OS_Name)

df["OS_Name"].replace({"HarmonyOS":"Harmony"},inplace=True)

df[df["os"].isna()][["index","model"]]

#### Extracting version of os 
##### like Android v4.4 then version = 4.4

In [ ]:
df["os_version"]  = df["os"].str.split("v").str.get(-1).str.strip().str.split().str.get(0)
df["os_version"]  = df["os_version"].str.split(".").str[:2].str.join(".")
df["os_version"].value_counts()


#### Some Card have half information like 
1. Memory Card (Hybrid)
2. Memory Card Supported
##### so by helping of gpt get full information and assign with their index

In [ ]:
memory_card_info  = {
    56: "Memory Card (Hybrid), upto 1 TB",
    74: "Memory Card (Hybrid), upto 1 TB",
    101: "Memory Card (Hybrid), upto 1 TB",
    104: "Memory Card (Hybrid), upto 1 TB",
    147: "Memory Card (Hybrid), upto 2 TB",
    165: "Memory Card Supported, upto 256 GB",
    176: "Memory Card Supported, upto 256 GB",
    228: "Memory Card Supported, upto 256 GB",
    235: "Memory Card Not Supported",
    256: "Memory Card Supported, upto 256 GB",
    278: "Memory Card Not Supported",
    284: "Memory Card Not Supported",
    287: "Memory Card (Hybrid), upto 1 TB",
    289: "Memory Card Supported, upto 1 TB",
    298: "Memory Card (Hybrid), upto 1 TB",
    318: "Memory Card Not Supported",
    327: "Memory Card Supported, upto 256 GB",
    330: "Memory Card (Hybrid), upto 512 GB",
    338: "Memory Card (Hybrid), upto 1 TB",
    343: "Memory Card Supported, upto 1 TB",
    344: "Memory Card Not Supported",
    362: "Memory Card (Hybrid), upto 1 TB",
    371: "Memory Card Supported, upto 1 TB",
    375: "Memory Card Supported, upto 128 GB",
    381: "Memory Card (Hybrid), upto 256 GB",
    406: "Memory Card Not Supported",
    409: "Memory Card (Hybrid), upto 256 GB",
    421: "Memory Card Not Supported",
    423: "Memory Card (Hybrid), upto 1 TB",
    433: "Memory Card Not Supported",
    435: "Memory Card (Hybrid), upto 256 GB",
    436: "Memory Card Supported, upto 256 GB",
    437: "Memory Card Supported, upto 1 TB",
    442: "Memory Card Supported, upto 1 TB",
    446: "Memory Card Not Supported",
    450: "Memory Card (Hybrid), upto 1 TB",
    451: "Memory Card Not Supported",
    453: "Memory Card (Hybrid), upto 1 TB",
    455: "Memory Card (Hybrid), upto 512 GB",
    457: "Memory Card (Hybrid), upto 1 TB",
    466: "Memory Card Not Supported",
    467: "Memory Card Supported, upto 512 GB",
    479: "Memory Card (Hybrid), upto 1 TB",
    490: "Memory Card Not Supported",
    497: "Memory Card (Hybrid), upto 1 TB",
    503: "Memory Card Not Supported",
    510: "Memory Card (Hybrid), upto 1 TB",
    514: "Memory Card Supported, upto 1 TB",
    516: "Memory Card Not Supported",
    527: "Memory Card Supported, upto 1 TB",
    540: "Memory Card Supported, upto 1 TB",
    550: "Memory Card Supported, upto 1 TB",
    551: "Memory Card Supported, upto 1 TB",
    555: "Memory Card Supported, upto 256 GB",
    561: "Memory Card (Hybrid), upto 1 TB",
    563: "Memory Card Supported, upto 1 TB",
    565: "Memory Card Supported, upto 1 TB",
    570: "Memory Card Supported, upto 1 TB",
    578: "Memory Card (Hybrid), upto 512 GB",
    584: "Memory Card Supported, upto 1 TB",
    585: "Memory Card (Hybrid), upto 1 TB",
    588: "Memory Card Supported, upto 1 TB",
    604: "Memory Card Supported, upto 256 GB",
    617: "Memory Card (Hybrid), upto 1 TB",
    621: "Memory Card Supported, upto 1 TB",
    622: "Memory Card Supported, upto 1 TB",
    636: "Memory Card Supported, upto 256 GB",
    642: "Memory Card Supported, upto 1 TB",
    650: "Memory Card Supported, upto 1 TB",
    652: "Memory Card Supported, upto 1 TB",
    660: "Memory Card Supported, upto 1 TB",
    672: "Memory Card (Hybrid), upto 256 GB",
    674: "Memory Card Not Supported",
    694: "Memory Card (Hybrid), upto 128 GB",
    704: "Memory Card Supported, upto 1 TB",
    707: "Memory Card Supported, upto 1 TB",
    718: "Memory Card Supported, upto 1 TB",
    727: "Memory Card Supported, upto 1 TB",
    732: "Memory Card (Hybrid), upto 1 TB",
    738: "Memory Card Supported, upto 1 TB",
    740: "Memory Card Supported, upto 512 GB",
    742: "Memory Card Supported, upto 1 TB",
    753: "Memory Card Supported, upto 256 GB",
    764: "Memory Card Supported, upto 2 TB",
    773: "Memory Card Supported, upto 1 TB",
    775: "Memory Card Supported, upto 512 GB",
    778: "Memory Card (Hybrid), upto 1 TB",
    783: "Memory Card Supported, upto 1 TB",
    787: "Memory Card Supported, upto 256 GB",
    790: "Memory Card Supported, upto 1 TB",
    791: "Memory Card Supported, upto 1 TB",
    823: "Memory Card Supported, upto 1 TB",
    825: "Memory Card Supported, upto 1 TB",
    835: "Memory Card Supported, upto 1 TB",
    843: "Memory Card (Hybrid), upto 512 GB",
    848: "Memory Card Supported, upto 512 GB",
    851: "Memory Card Supported, upto 1 TB",
    853: "Memory Card Supported, upto 512 GB",
    854: "Memory Card Supported, upto 512 GB",
    858: "Memory Card Supported, upto 1 TB",
    859: "Memory Card Supported, upto 128 GB",
    872: "Memory Card Supported, upto 512 GB",
    877: "Memory Card Supported, upto 1 TB",
    887: "Memory Card Supported, upto 256 GB",
    888: "Memory Card Supported, upto 1 TB",
    890: "Memory Card (Hybrid), upto 256 GB",
    892: "Memory Card Supported, upto 256 GB",
    896: "Memory Card Supported, upto 512 GB",
    897: "Memory Card Supported, upto 1 TB",
    915: "Memory Card Supported, upto 512 GB",
    922: "Memory Card Supported, upto 512 GB",
    924: "Memory Card Supported, upto 1 TB",
    929: "Memory Card Supported, upto 256 GB",
    938: "Memory Card Supported, upto 512 GB",
    941: "Memory Card Supported, upto 256 GB",
    954: "Memory Card Supported, upto 256 GB",
    964: "Memory Card Supported, upto 256 GB",
    969: "Memory Card Supported, upto 128 GB",
    973: "Memory Card Supported, upto 512 GB",
    977: "Memory Card Supported, upto 256 GB"
}

for  idx, card_info in memory_card_info.items():
    mask = df['index'] == idx
    df.loc[mask, 'card'] = card_info

#### If card have memory
##### Then give it 1 else 0

In [ ]:
df["card"].fillna("Memory Card Not Supported",inplace=True)
df["card_support"] = (df["card"].str.contains("Not")).apply(lambda x : 1 if x is False else 0)
df["card_support"].value_counts()


#### Extract  card Slot type
1. Hybird
2. Supported == Dedicated
3. If card not have Not in it (Memory Card Not supported) simply give None slot 

In [ ]:
df["slot_type"] = df[~df["card"].str.contains("Not")]["card"].str.split(",").str.get(0).str.strip().str.split().str.get(-1)

print(df["slot_type"].isnull().sum())   ### These are slot that donot have memory card 
print(df["slot_type"].value_counts())
df["slot_type"].fillna("None",inplace=True)  ## Fill with None

slot = {"(Hybrid)" : "Hybird",
        "Supported": "Dedicated"}

df["slot_type"] = df["slot_type"].map(slot,inplace=True)
df["slot_type"].value_counts()

### Extracting card dedicated capacity and convert all into Gb like 1 TB = 1024 Gb

In [ ]:
df["capacity_gb"] =df["card"].str.split(",").str.get(1).str.strip().str.split("upto").str.get(-1).str.strip()


def extract(value):
        if pd.isna(value):
          return 0
        parts = str(value).split()
        if "TB" in str(value):
        # Extract number and convert TB to GB
          number = int(parts[0])
          return number * 1024
        elif "GB" in str(value):
        # Extract just the number
         return int(parts[0])
        else:
          return 0

    
df["capacity_gb"] = df["capacity_gb"].apply(extract)

In [ ]:
df["capacity_gb"].value_counts()

#### Convert Core into numerical value

In [ ]:
df["Processor_Core"].value_counts().index
df["Processor_Core"] = df["Processor_Core"].replace({'Octa Core' : 8, 'Hexa Core' : 6, 'Quad Core' : 4},inplace=True)

### Dropping Columns Because we already have extract information

In [ ]:
df.drop(columns= ["os","card","camera","index",'processor',"ram","battery","display"],inplace=True)

df["slot_type"] =df["slot_type"].fillna("Not Slot",inplace=True)

#### Storing cleaned data set

In [ ]:


df["Num_Back_Cam"].value_counts()

In [ ]:
df["rating"] = df["rating"].replace({"No Rating":0},inplace=True)
df["rating"].value_counts()

## Final Step — Drop Leaky Column & Export

`rating` is a post-sale user review score — drop it to prevent data leakage.

In [ ]:
# Drop rating — post-sale signal, not available at inference time
df.drop(columns=["rating"], errors="ignore", inplace=True)
df.drop(columns=["index", "processor"], errors="ignore", inplace=True)
print(f"Final shape: {df.shape}")
df.head(2)


In [ ]:
df.to_csv("data\Clean2_Smart_Phones.csv")